<a href="https://colab.research.google.com/github/venkat4246/zepto-ai-ml-capstone-project/blob/main/data_pipeline/01_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from urllib.parse import urljoin

BASE_URL = "https://books.toscrape.com/"
books = []

headers = {
    "User-Agent": "Mozilla/5.0"
}

for page in range(1, 6):

    page_url = urljoin(
        BASE_URL,
        f"catalogue/page-{page}.html"
    )

    response = requests.get(
        page_url,
        headers=headers,
        timeout=20
    )

    print(f"Page {page}: HTTP {response.status_code}")

    soup = BeautifulSoup(response.text, "html.parser")

    products = soup.select("article.product_pod")

    for product in products:

        title_tag = product.select_one("h3 a")

        title = title_tag.get("title", "").strip()

        price_tag = product.select_one(".price_color")
        price = price_tag.get_text(strip=True)

        rating_tag = product.select_one(".star-rating")

        star_rating = "Unknown"

        if rating_tag:
            for rating in ["One", "Two", "Three", "Four", "Five"]:
                if rating in rating_tag.get("class", []):
                    star_rating = rating
                    break

        availability_tag = product.select_one(".availability")

        availability = (
            availability_tag.get_text(" ", strip=True)
            if availability_tag
            else "Unknown"
        )

        # Get book detail page
        detail_url = urljoin(
            page_url,
            title_tag.get("href")
        )

        detail_response = requests.get(
            detail_url,
            headers=headers,
            timeout=20
        )

        category = "Unknown"

        if detail_response.status_code == 200:

            detail_soup = BeautifulSoup(
                detail_response.text,
                "html.parser"
            )

            breadcrumb = detail_soup.select(
                "ul.breadcrumb li"
            )

            # Home -> Books -> Category -> Book
            if len(breadcrumb) >= 3:

                category = breadcrumb[2].get_text(
                    strip=True
                )

        books.append({
            "title": title,
            "price": price,
            "star_rating": star_rating,
            "availability": availability,
            "category": category
        })

        time.sleep(0.05)

    time.sleep(0.5)


# Create DataFrame
df_raw = pd.DataFrame(books)

# Remove duplicates
df_raw = df_raw.drop_duplicates(
    subset=["title"]
).reset_index(drop=True)


print("\n==============================")
print("SCRAPING COMPLETED")
print("==============================")

print("Total books scraped:", len(df_raw))

print("\nColumns:")
print(df_raw.columns.tolist())

print("\nCategory counts:")
print(df_raw["category"].value_counts())

print("\nNumber of categories:")
print(df_raw["category"].nunique())

print("\nFirst 10 records:")
display(df_raw.head(10))


# Save CSV
df_raw.to_csv(
    "books_raw.csv",
    index=False
)

print("\nRaw data saved as: books_raw.csv")